# Geometry-MMALS G1 v1.0.4
## Matched Initialization, Matched Compute, and Retention Anchors

**Scientific status:** controlled protocol implementation and development experiment.

**Accepted claim:** C0 pipeline/protocol integrity only.

**Not claimed:** C1-C6 qualification, quantum advantage, backward transfer, replay-free continual learning, or domain-general geometry.

v1.0.4 follows the first informative cross-angle run in v1.0.3. The previous run showed that same-source paired supervision can strengthen context and route ordering, but the comparison was confounded by different MMALS initializations and unequal forward-compute budgets. It also improved plasticity while increasing trained-angle forgetting.

This notebook therefore asks three narrower questions:

1. Does paired geometry still improve route/context order when the paired baseline has exactly the same inputs, forwards, optimizer steps, initialization, and source order?
2. Can a small old-angle classification anchor preserve the geometric gain while reducing forgetting?
3. Do the resulting effects propagate beyond route geometry into interpolation, synthesis, host function, and causal specificity?

## Research history

| Version | Main question | Outcome |
|---|---|---|
| v1.0.0 | Can functional MMALS geometry be specified and falsified? | Article, evidence ladder, and C0 scaffold. |
| v1.0.1 | Can claims, gates, notation, and dual-memory contracts be audited? | C0-only status and pilot gates formalized. |
| v1.0.2 | Can the Fisher-Rao route loss and Colab pipeline run without NaNs? | Numerical stability passed; finite traces obtained. |
| v1.0.3 | Does the optimization unit contain genuine cross-angle evidence? | Same-source paired protocol introduced; first informative development result. |
| **v1.0.4** | Does geometry add value under matched initialization and compute, and can a retention anchor reduce forgetting? | **Implemented here; no qualification claim before multi-seed runs.** |

### v1.0.3 development evidence motivating this revision

The seed-0 development run reported stronger source-level order for the paired condition in context and route spaces, higher mean held-out interpolation accuracy, and higher final trained-angle accuracy. However, forgetting increased, causal specificity remained below one, and paired training used more image-forwards than the current-only reference.

These observations are treated as **hypothesis-generating development evidence**, not as passed gates.

## Tracked changes in v1.0.4

- **CHG-104-01 - Shared MMALS initialization:** every method loads one immutable initial state.
- **CHG-104-02 - Deterministic method-independent source order:** stage/epoch loader seeds do not depend on method.
- **CHG-104-03 - Compute-matched paired control:** paired multi-angle batches with geometry weight zero.
- **CHG-104-04 - Explicit attribution contrast:** `paired_geometry` is compared primarily with `paired_compute_matched_no_geometry`.
- **CHG-104-05 - Retention-anchor variant:** old-angle classification evidence is added without extra forwards.
- **CHG-104-06 - Forward/step accounting:** images forwarded, source examples, optimizer steps, and wall time are exported.
- **CHG-104-07 - Sensory acceptance gate:** qualification mode stops if the frozen sensory grove is below threshold.
- **CHG-104-08 - Initial-state and split digests:** immutable hashes are recorded in the manifest.
- **CHG-104-09 - Budget assertions:** paired variants must have identical per-stage forwards and optimizer steps.
- **CHG-104-10 - Attribution and retention deltas:** geometry effect and anchor effect are summarized separately.
- **CHG-104-11 - Clean archival export:** notebook, CSVs, manifest, and a separate PDF report are the evidence bundle.
- **CHG-104-12 - Strict non-claims:** development results cannot automatically promote C1-C6.

## 0. Setup

The canonical Python package remains `geometry_mmalls` from the standalone `geometry-mmalls-g1` repository. Do not install the TPUT reference copy in the same runtime.

In [ ]:
import os
import sys
import shutil
import importlib
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"

# Set True after pushing a newer package revision to GitHub.
FORCE_REFRESH = False

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "pyproject.toml").exists(), f"Missing pyproject.toml in {REPO_DIR}"
assert (SRC_DIR / "geometry_mmalls" / "__init__.py").exists(), "Missing geometry_mmalls package"

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)

src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

import geometry_mmalls

loaded_from = Path(geometry_mmalls.__file__).resolve()
expected_root = (SRC_DIR / "geometry_mmalls").resolve()
assert loaded_from.parent == expected_root, f"Wrong geometry_mmalls package: {loaded_from}"

print("Python:", sys.executable)
print("Repository:", REPO_DIR)
print("geometry_mmalls:", loaded_from)
print("package version:", geometry_mmalls.__version__)

In [ ]:
from pathlib import Path
import copy
import hashlib
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import fisher_rao_distance, paired_route_geometry_loss
from geometry_mmalls.metrics import bootstrap_mean_ci, centroid_geometry_scores, grouped_geometry_scores
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder

NOTEBOOK_VERSION = "1.0.4"
config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())

RUN_MODE = "development"  # "smoke", "development", or "qualification"
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Core method set. The three paired variants have equal image-forward budgets.
METHOD_SPECS = {
    "current_only_no_geometry": {
        "paired": False,
        "geometry_weight": 0.0,
        "anchor_weight": 0.0,
    },
    "paired_compute_matched_no_geometry": {
        "paired": True,
        "geometry_weight": 0.0,
        "anchor_weight": 0.0,
    },
    "paired_geometry": {
        "paired": True,
        "geometry_weight": float(config["losses"]["route_geometry"]),
        "anchor_weight": 0.0,
    },
    "paired_geometry_anchor_010": {
        "paired": True,
        "geometry_weight": float(config["losses"]["route_geometry"]),
        "anchor_weight": 0.10,
    },
}
METHODS = list(METHOD_SPECS)
PAIRED_METHODS = [name for name, spec in METHOD_SPECS.items() if spec["paired"]]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_angles = list(map(float, config["data"]["train_angles"]))
interp_angles = list(map(float, config["data"]["interpolation_angles"]))
extra_angles = list(map(float, config["data"]["extrapolation_angles"]))
all_eval_angles = train_angles + interp_angles + extra_angles

if RUN_MODE == "smoke":
    TRAIN_SOURCE_LIMIT = 64
    TEST_SOURCE_LIMIT = 64
    SENSORY_SOURCE_LIMIT = 256
    SENSORY_TEST_LIMIT = 128
    SENSORY_EPOCHS = 1
    STAGE_EPOCHS = 1
    BOOTSTRAP_SAMPLES = 100
    SOURCE_BATCH_SIZE = 32
    SENSORY_GATE = 0.0
elif RUN_MODE == "development":
    TRAIN_SOURCE_LIMIT = 512
    TEST_SOURCE_LIMIT = 256
    SENSORY_SOURCE_LIMIT = 2000
    SENSORY_TEST_LIMIT = 1000
    SENSORY_EPOCHS = 2
    STAGE_EPOCHS = 2
    BOOTSTRAP_SAMPLES = 500
    SOURCE_BATCH_SIZE = 64
    SENSORY_GATE = 0.85
elif RUN_MODE == "qualification":
    TRAIN_SOURCE_LIMIT = int(config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT = 2000
    SENSORY_SOURCE_LIMIT = min(12000, 60000)
    SENSORY_TEST_LIMIT = 5000
    SENSORY_EPOCHS = int(config["training"]["sensory_pretrain_epochs"])
    STAGE_EPOCHS = int(config["training"]["stage_epochs"])
    BOOTSTRAP_SAMPLES = int(config["metrics"]["bootstrap_samples"])
    SOURCE_BATCH_SIZE = int(config["paired_protocol"]["source_batch_size"])
    SENSORY_GATE = 0.95
else:
    raise ValueError("RUN_MODE must be smoke, development, or qualification")

print({
    "notebook_version": NOTEBOOK_VERSION,
    "run_mode": RUN_MODE,
    "device": str(DEVICE),
    "methods": METHODS,
    "train_source_limit": TRAIN_SOURCE_LIMIT,
    "test_source_limit": TEST_SOURCE_LIMIT,
    "sensory_source_limit": SENSORY_SOURCE_LIMIT,
})

## 1. Numerical and structural self-tests

Training uses stable chordal distance in square-root simplex coordinates. Fisher-Rao distance remains the evaluation metric. The self-test also verifies that a zero geometry weight remains differentiable.

In [ ]:
_probe_logits = torch.randn(8, 5, 4, requires_grad=True)
_probe_routes = torch.softmax(_probe_logits, dim=-1)
_probe_angles = torch.tensor([-60.0, -30.0, 0.0, 30.0, 60.0])
_probe_loss = paired_route_geometry_loss(_probe_routes, _probe_angles)
_probe_loss.backward()
assert torch.isfinite(_probe_loss)
assert _probe_logits.grad is not None and torch.isfinite(_probe_logits.grad).all()

_zero = (_probe_routes.sum() * 0.0)
assert torch.isfinite(_zero)
print("Paired route-geometry backward gate: PASS", float(_probe_loss.detach()))

## 2. Controlled source split and deterministic loaders

The same source image identifiers are used at every angle. Interpolation and extrapolation angles never enter training or checkpoint selection. Loader seeds depend only on the global seed, stage, and epoch - never on the method name.

In [ ]:
root = Path(config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)

rng = np.random.default_rng(SEED)
train_perm = rng.permutation(len(base_train))
test_perm = rng.permutation(len(base_test))

sensory_indices = train_perm[:SENSORY_SOURCE_LIMIT].tolist()
train_indices = train_perm[:TRAIN_SOURCE_LIMIT].tolist()
test_indices = test_perm[:TEST_SOURCE_LIMIT].tolist()
sensory_test_indices = test_perm[:SENSORY_TEST_LIMIT].tolist()


def sequence_checksum(values):
    payload = ",".join(map(str, values)).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def make_generator(seed):
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return generator


def fixed_loader(angle, train, indices, shuffle, loader_seed=0, batch_size=128):
    ds = FixedAngleMNIST(root, angle=angle, train=train, download=True)
    ds = Subset(ds, indices)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=make_generator(loader_seed) if shuffle else None,
    )


def multi_loader(angles, train, indices, shuffle, loader_seed=0):
    ds = MultiAngleMNIST(
        root,
        angles=angles,
        train=train,
        indices=indices,
        download=True,
    )
    return DataLoader(
        ds,
        batch_size=SOURCE_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        generator=make_generator(loader_seed) if shuffle else None,
    )


def epoch_loader_seed(stage, epoch):
    return SEED * 1_000_000 + stage * 10_000 + epoch

split_manifest = {
    "seed": SEED,
    "sensory_source_count": len(sensory_indices),
    "train_source_count": len(train_indices),
    "test_source_count": len(test_indices),
    "sensory_index_sha256": sequence_checksum(sensory_indices),
    "train_index_sha256": sequence_checksum(train_indices),
    "test_index_sha256": sequence_checksum(test_indices),
}
split_manifest

## 3. Pretrain, evaluate, and freeze the sensory grove

The sensory grove is a control boundary. Qualification mode requires a minimum 0-degree test accuracy before any Geometry-MMALS gate may be considered. Development mode records a warning instead of aborting.

In [ ]:
latent_dim = int(config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
sensory_optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(sensory_head.parameters()),
    lr=float(config["training"]["learning_rate"]),
    weight_decay=float(config["training"]["weight_decay"]),
)

sensory_history = []
for epoch in range(SENSORY_EPOCHS):
    sensory_loader = fixed_loader(
        0.0,
        True,
        sensory_indices,
        True,
        loader_seed=SEED * 1000 + epoch,
        batch_size=128,
    )
    encoder.train()
    sensory_head.train()
    total = correct = 0
    loss_sum = 0.0
    for x, y, _, _ in sensory_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x))
        loss = F.cross_entropy(logits, y)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite sensory loss")
        sensory_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(sensory_head.parameters()), 5.0
        )
        sensory_optimizer.step()
        total += y.numel()
        correct += (logits.argmax(1) == y).sum().item()
        loss_sum += float(loss.detach()) * y.numel()
    row = {"epoch": epoch, "loss": loss_sum / total, "train_accuracy": correct / total}
    sensory_history.append(row)
    print(row)

@torch.no_grad()
def evaluate_sensory(angle=0.0):
    encoder.eval()
    sensory_head.eval()
    loader = fixed_loader(angle, False, sensory_test_indices, False, batch_size=256)
    total = correct = 0
    loss_sum = 0.0
    for x, y, _, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x))
        loss_sum += float(F.cross_entropy(logits, y, reduction="sum"))
        total += y.numel()
        correct += (logits.argmax(1) == y).sum().item()
    return {"angle": angle, "accuracy": correct / total, "nll": loss_sum / total}

sensory_gate_result = evaluate_sensory(0.0)
print("Sensory gate:", sensory_gate_result, "threshold:", SENSORY_GATE)
if RUN_MODE == "qualification" and sensory_gate_result["accuracy"] < SENSORY_GATE:
    raise RuntimeError("Qualification blocked: frozen sensory grove is below acceptance threshold")
if sensory_gate_result["accuracy"] < SENSORY_GATE:
    print("DEVELOPMENT WARNING: sensory gate not reached; no C1-C6 promotion is allowed.")

sensory_state = copy.deepcopy(encoder.state_dict())

## 4. Shared MMALS initialization and immutable state hash

All methods load exactly the same context network, router, hosts, synthesis normalization, classifier, and frozen encoder state. This removes the main initialization confound from v1.0.3.

In [ ]:
def build_model():
    local_encoder = SmallConvEncoder(latent_dim).to(DEVICE)
    local_encoder.load_state_dict(sensory_state)
    return GeometryMMALS(
        local_encoder,
        latent_dim=latent_dim,
        context_dim=int(config["model"]["context_dim"]),
        num_hosts=int(config["model"]["num_hosts"]),
        host_hidden_dim=int(config["model"]["host_hidden_dim"]),
        freeze_encoder=True,
    ).to(DEVICE)


def state_dict_sha256(state_dict):
    digest = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("utf-8"))
        digest.update(str(tuple(tensor.shape)).encode("utf-8"))
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

# Build the initial MMALS state exactly once.
torch.manual_seed(SEED + 104)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED + 104)
base_model = build_model()
INITIAL_MMALS_STATE = copy.deepcopy(base_model.state_dict())
INITIAL_MMALS_SHA256 = state_dict_sha256(INITIAL_MMALS_STATE)
del base_model

print("Initial MMALS SHA256:", INITIAL_MMALS_SHA256)

## 5. Continual training variants

### Primary attribution contrast

`paired_compute_matched_no_geometry` versus `paired_geometry`

Both variants receive the same same-source multi-angle tensors, execute the same image forwards, use the same number of optimizer steps, and start from the same parameters. Their only intended difference is the paired route-geometry term.

### Retention contrast

`paired_geometry` versus `paired_geometry_anchor_010`

Both have matched compute and geometry. The anchor variant adds a 0.10-weighted classification loss on previously seen angle views already present in the paired batch. It performs no additional forward pass and is not claimed to be replay-free.

In [ ]:
def host_diversity(host_outputs):
    h = F.normalize(host_outputs, dim=-1)
    sim = torch.einsum("bhd,bjd->bhj", h, h)
    mask = ~torch.eye(sim.shape[-1], dtype=torch.bool, device=sim.device)
    return sim[:, mask].square().mean()


def assert_finite_trace(trace, where):
    tensors = {
        "z0": trace.z0,
        "context": trace.context,
        "route": trace.route,
        "host_outputs": trace.host_outputs,
        "z_mm": trace.z_mm,
        "logits": trace.logits,
    }
    bad = [name for name, value in tensors.items() if not torch.isfinite(value).all()]
    if bad:
        raise FloatingPointError(f"Non-finite tensors at {where}: {bad}")


@torch.no_grad()
def evaluate_model(model, angles, stage, method):
    rows = []
    model.eval()
    for angle in angles:
        loader = fixed_loader(angle, False, test_indices, False, batch_size=256)
        total = correct = 0
        nll_sum = 0.0
        for x, y, _, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = model(x)
            assert_finite_trace(trace, f"eval method={method}, angle={angle}")
            nll = F.cross_entropy(trace.logits, y, reduction="sum")
            total += y.numel()
            correct += (trace.logits.argmax(1) == y).sum().item()
            nll_sum += float(nll)
        rows.append({
            "method": method,
            "stage": stage,
            "angle": angle,
            "angle_type": (
                "train" if angle in train_angles
                else "interpolation" if angle in interp_angles
                else "extrapolation"
            ),
            "accuracy": correct / total,
            "nll": nll_sum / total,
        })
    return rows


def train_method(method):
    spec = METHOD_SPECS[method]
    model = build_model()
    model.load_state_dict(INITIAL_MMALS_STATE, strict=True)
    assert state_dict_sha256(model.state_dict()) == INITIAL_MMALS_SHA256

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=float(config["training"]["learning_rate"]),
        weight_decay=float(config["training"]["weight_decay"]),
    )

    stage_rows = []
    evaluation_rows = []

    for stage, current_angle in enumerate(train_angles):
        seen_angles = train_angles[: stage + 1]
        totals = {
            "source_examples": 0,
            "forward_images": 0,
            "optimizer_steps": 0,
            "loss": 0.0,
            "current_ce": 0.0,
            "anchor_ce": 0.0,
            "geo": 0.0,
            "div": 0.0,
        }
        stage_start = time.perf_counter()

        model.train()
        for epoch in range(STAGE_EPOCHS):
            loader_seed = epoch_loader_seed(stage, epoch)
            if spec["paired"]:
                loader = multi_loader(seen_angles, True, train_indices, True, loader_seed)
            else:
                loader = fixed_loader(current_angle, True, train_indices, True, loader_seed, 128)

            for batch_id, batch in enumerate(loader):
                if spec["paired"]:
                    views, y, factors, _ = batch
                    source_count, angle_count = views.shape[:2]
                    flat_views = views.reshape(-1, *views.shape[2:]).to(DEVICE)
                    y = y.to(DEVICE)
                    factors = factors.to(DEVICE)

                    trace = model(flat_views)
                    assert_finite_trace(trace, f"{method}, stage={stage}, epoch={epoch}, batch={batch_id}")
                    logits = trace.logits.reshape(source_count, angle_count, -1)
                    routes = trace.route.reshape(source_count, angle_count, -1)
                    hosts = trace.host_outputs.reshape(
                        source_count, angle_count, trace.host_outputs.shape[1], -1
                    )

                    current_index = angle_count - 1
                    current_ce = F.cross_entropy(logits[:, current_index], y)
                    if angle_count > 1:
                        old_logits = logits[:, :-1].reshape(-1, logits.shape[-1])
                        old_targets = y[:, None].expand(-1, angle_count - 1).reshape(-1)
                        anchor_ce = F.cross_entropy(old_logits, old_targets)
                    else:
                        anchor_ce = logits.sum() * 0.0

                    geo = paired_route_geometry_loss(
                        routes,
                        factors,
                        bandwidth=float(config["losses"]["route_bandwidth_degrees"]),
                        far_margin=float(config["losses"]["paired_route_far_margin"]),
                        far_weight=float(config["losses"]["paired_route_far_weight"]),
                        match_weight=float(config["losses"]["paired_route_match_weight"]),
                    )
                    div = host_diversity(hosts[:, current_index])
                    forward_images = source_count * angle_count
                else:
                    x, y, _, _ = batch
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    trace = model(x)
                    assert_finite_trace(trace, f"{method}, stage={stage}, epoch={epoch}, batch={batch_id}")
                    current_ce = F.cross_entropy(trace.logits, y)
                    anchor_ce = trace.logits.sum() * 0.0
                    geo = trace.route.sum() * 0.0
                    div = host_diversity(trace.host_outputs)
                    source_count = y.numel()
                    forward_images = source_count

                loss = (
                    float(config["losses"]["classification"]) * current_ce
                    + float(spec["anchor_weight"]) * anchor_ce
                    + float(spec["geometry_weight"]) * geo
                    + float(config["losses"]["host_functional_diversity"]) * div
                )

                if not torch.isfinite(torch.stack([current_ce, anchor_ce, geo, div, loss])).all():
                    raise FloatingPointError(
                        f"Non-finite loss in {method}, stage={stage}: "
                        f"current_ce={current_ce.item()}, anchor_ce={anchor_ce.item()}, "
                        f"geo={geo.item()}, div={div.item()}"
                    )

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 5.0
                )
                if not torch.isfinite(torch.as_tensor(grad_norm)):
                    raise FloatingPointError("Non-finite gradient norm")
                optimizer.step()

                totals["source_examples"] += source_count
                totals["forward_images"] += forward_images
                totals["optimizer_steps"] += 1
                totals["loss"] += float(loss.detach()) * source_count
                totals["current_ce"] += float(current_ce.detach()) * source_count
                totals["anchor_ce"] += float(anchor_ce.detach()) * source_count
                totals["geo"] += float(geo.detach()) * source_count
                totals["div"] += float(div.detach()) * source_count

        count = max(totals["source_examples"], 1)
        stage_row = {
            "method": method,
            "stage": stage,
            "current_angle": current_angle,
            "seen_angles": json.dumps(seen_angles),
            "paired": bool(spec["paired"]),
            "geometry_weight": float(spec["geometry_weight"]),
            "anchor_weight": float(spec["anchor_weight"]),
            "source_examples": totals["source_examples"],
            "forward_images": totals["forward_images"],
            "optimizer_steps": totals["optimizer_steps"],
            "wall_seconds": time.perf_counter() - stage_start,
            "loss": totals["loss"] / count,
            "current_ce": totals["current_ce"] / count,
            "anchor_ce": totals["anchor_ce"] / count,
            "geo": totals["geo"] / count,
            "div": totals["div"] / count,
        }
        stage_rows.append(stage_row)
        print(stage_row)
        evaluation_rows.extend(evaluate_model(model, all_eval_angles, stage, method))

    return model, pd.DataFrame(stage_rows), pd.DataFrame(evaluation_rows)

In [ ]:
models = {}
stage_tables = []
evaluation_tables = []

for method in METHODS:
    print("\n=== TRAINING", method, "===")
    model, method_stage_df, method_evaluation_df = train_method(method)
    models[method] = model
    stage_tables.append(method_stage_df)
    evaluation_tables.append(method_evaluation_df)

stage_df = pd.concat(stage_tables, ignore_index=True)
evaluation_df = pd.concat(evaluation_tables, ignore_index=True)
stage_df

## 6. Compute-budget audit

The primary paired variants must have identical per-stage image-forward counts and optimizer-step counts. The current-only reference is intentionally reported separately because it is cheaper.

In [ ]:
paired_budget = stage_df[stage_df.method.isin(PAIRED_METHODS)][
    ["method", "stage", "forward_images", "optimizer_steps", "source_examples"]
].copy()

for stage in sorted(paired_budget.stage.unique()):
    block = paired_budget[paired_budget.stage == stage]
    assert block.forward_images.nunique() == 1, f"Paired forward budget mismatch at stage {stage}"
    assert block.optimizer_steps.nunique() == 1, f"Paired optimizer-step mismatch at stage {stage}"
    assert block.source_examples.nunique() == 1, f"Paired source budget mismatch at stage {stage}"

compute_summary_df = (
    stage_df.groupby("method")
    .agg(
        total_forward_images=("forward_images", "sum"),
        total_optimizer_steps=("optimizer_steps", "sum"),
        total_source_examples=("source_examples", "sum"),
        total_wall_seconds=("wall_seconds", "sum"),
    )
    .reset_index()
)
print("Matched paired compute audit: PASS")
compute_summary_df

## 7. Staged accuracy and forgetting

Forgetting is computed only after an angle has been introduced. Held-out angles are evaluated for interpolation and extrapolation but are never treated as learned tasks.

In [ ]:
def forgetting_table(evaluation):
    rows = []
    final_stage = int(evaluation.stage.max())
    for method in evaluation.method.unique():
        sub = evaluation[(evaluation.method == method) & (evaluation.angle_type == "train")]
        for angle in train_angles:
            seen_stage = train_angles.index(angle)
            angle_rows = sub[(sub.angle == angle) & (sub.stage >= seen_stage)].sort_values("stage")
            best = float(angle_rows.accuracy.max())
            final = float(angle_rows[angle_rows.stage == final_stage].accuracy.iloc[0])
            rows.append({
                "method": method,
                "angle": angle,
                "best_accuracy": best,
                "final_accuracy": final,
                "forgetting": best - final,
            })
    return pd.DataFrame(rows)

forgetting_df = forgetting_table(evaluation_df)
forgetting_df.groupby("method").agg(
    mean_forgetting=("forgetting", "mean"),
    final_trained_accuracy=("final_accuracy", "mean"),
)

## 8. Same-source paired trace collection

Every trace retains the original source index. Geometry evidence is computed within source blocks and then bootstrapped across source images.

In [ ]:
@torch.no_grad()
def collect_paired_trace(model, method, angles, source_indices):
    model.eval()
    rows = []
    loader = multi_loader(angles, False, source_indices, False)
    for views, labels, factors, source_ids in loader:
        source_count, angle_count = views.shape[:2]
        flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
        trace = model(flat)
        assert_finite_trace(trace, f"paired trace {method}")

        z0 = trace.z0.reshape(source_count, angle_count, -1).cpu().numpy()
        context = trace.context.reshape(source_count, angle_count, -1).cpu().numpy()
        route = trace.route.reshape(source_count, angle_count, -1).cpu().numpy()
        z_mm = trace.z_mm.reshape(source_count, angle_count, -1).cpu().numpy()
        logits = trace.logits.reshape(source_count, angle_count, -1).cpu().numpy()
        hosts = trace.host_outputs.reshape(
            source_count, angle_count, trace.host_outputs.shape[1], -1
        ).cpu().numpy()

        for b in range(source_count):
            for a in range(angle_count):
                rows.append({
                    "method": method,
                    "source_index": int(source_ids[b]),
                    "label": int(labels[b]),
                    "angle": float(factors[b, a]),
                    "prediction": int(np.argmax(logits[b, a])),
                    "z0": z0[b, a],
                    "context": context[b, a],
                    "route": route[b, a],
                    "z_mm": z_mm[b, a],
                    "hosts": hosts[b, a],
                })
    return pd.DataFrame(rows)

trace_df = pd.concat(
    [collect_paired_trace(model, method, all_eval_angles, test_indices) for method, model in models.items()],
    ignore_index=True,
)
print("paired trace rows:", len(trace_df))

## 9. Source-block and centroid geometry

Source-block scores test local same-image trajectories. Factor-centroid scores test whether those local trajectories align into a common global ordering. No global all-pairs p-value is used as evidence.

In [ ]:
def stack_column(frame, name):
    return np.stack(frame[name].to_numpy())

geometry_rows = []
for method in METHODS:
    sub = trace_df[trace_df.method == method]
    factors = sub.angle.to_numpy(float)
    groups = sub.source_index.to_numpy()
    spaces = {
        "sensory": ("euclidean", stack_column(sub, "z0")),
        "context": ("euclidean", stack_column(sub, "context")),
        "route": ("fisher_rao", stack_column(sub, "route")),
        "synthesis": ("euclidean", stack_column(sub, "z_mm")),
    }
    for space, (metric, reps) in spaces.items():
        grouped = grouped_geometry_scores(factors, reps, groups, metric=metric)
        rho_mean, rho_low, rho_high = bootstrap_mean_ci(
            grouped["rho"], samples=BOOTSTRAP_SAMPLES, seed=SEED
        )
        stress_mean, stress_low, stress_high = bootstrap_mean_ci(
            grouped["stress"], samples=BOOTSTRAP_SAMPLES, seed=SEED + 1
        )
        centroid = centroid_geometry_scores(factors, reps, metric=metric)
        geometry_rows.append({
            "method": method,
            "space": space,
            "source_rho_mean": rho_mean,
            "source_rho_ci_low": rho_low,
            "source_rho_ci_high": rho_high,
            "source_stress_mean": stress_mean,
            "source_stress_ci_low": stress_low,
            "source_stress_ci_high": stress_high,
            "centroid_rho": centroid["rho"],
            "centroid_stress": centroid["stress"],
            "source_blocks": len(grouped["rho"]),
            "status": "development_not_qualified",
        })

geometry_df = pd.DataFrame(geometry_rows)
geometry_df

## 10. Held-out interpolation controls

The primary interpolation control compares the actual held-out route centroid with interpolation between adjacent trained centroids in square-root simplex coordinates and with the nearest trained route.

In [ ]:
def centroid_by_angle(frame, column):
    return {
        float(angle): np.stack(group[column].to_numpy()).mean(axis=0)
        for angle, group in frame.groupby("angle")
    }


def linear_neighbors(angle, trained):
    lower = max(value for value in trained if value < angle)
    upper = min(value for value in trained if value > angle)
    alpha = (angle - lower) / (upper - lower)
    return lower, upper, alpha

interpolation_rows = []
final_stage = int(evaluation_df.stage.max())
for method in METHODS:
    final_eval = evaluation_df[
        (evaluation_df.method == method)
        & (evaluation_df.stage == final_stage)
        & (evaluation_df.angle_type == "interpolation")
    ]
    sub = trace_df[trace_df.method == method]
    context_centroids = centroid_by_angle(sub, "context")
    route_centroids = centroid_by_angle(sub, "route")

    for row in final_eval.itertuples():
        angle = float(row.angle)
        lower, upper, alpha = linear_neighbors(angle, train_angles)

        c_pred = (1.0 - alpha) * context_centroids[lower] + alpha * context_centroids[upper]
        context_error = float(np.linalg.norm(context_centroids[angle] - c_pred))

        p_low = route_centroids[lower]
        p_high = route_centroids[upper]
        root_pred = (
            (1.0 - alpha) * np.sqrt(np.clip(p_low, 1e-12, None))
            + alpha * np.sqrt(np.clip(p_high, 1e-12, None))
        )
        p_pred = np.square(root_pred)
        p_pred = p_pred / p_pred.sum()
        actual = route_centroids[angle]
        route_interp_error = fisher_rao_distance(actual, p_pred)
        nearest_error = min(
            fisher_rao_distance(actual, p_low),
            fisher_rao_distance(actual, p_high),
        )

        interpolation_rows.append({
            "method": method,
            "angle": angle,
            "accuracy": float(row.accuracy),
            "nll": float(row.nll),
            "context_interpolation_error": context_error,
            "route_interpolation_error": route_interp_error,
            "nearest_route_error": nearest_error,
            "route_interpolation_beats_nearest": route_interp_error < nearest_error,
            "status": "development_not_qualified",
        })

interpolation_df = pd.DataFrame(interpolation_rows)
interpolation_df

## 11. Host ablation and route-mass bundle

A host is not specialized merely because it receives route mass. The bundle reports route mass, mean and median ablation impact, and the fraction of examples with positive contribution.

In [ ]:
@torch.no_grad()
def host_bundle(model, method, angles, source_indices):
    rows = []
    model.eval()
    for angle in angles:
        loader = fixed_loader(angle, False, source_indices, False, batch_size=256)
        for x, y, _, source_id in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = model(x)
            assert_finite_trace(trace, f"host bundle {method}, angle={angle}")
            base = F.log_softmax(trace.logits, -1).gather(1, y[:, None]).squeeze(1)
            for host in range(trace.route.shape[1]):
                route = trace.route.clone()
                route[:, host] = 0.0
                route = route / route.sum(1, keepdim=True).clamp_min(1e-8)
                z = torch.einsum("bh,bhd->bd", route, trace.host_outputs)
                logits = model.classifier(model.synthesis_norm(z))
                ablated = F.log_softmax(logits, -1).gather(1, y[:, None]).squeeze(1)
                impact = (base - ablated).detach().cpu().numpy()
                mass = trace.route[:, host].detach().cpu().numpy()
                for idx, value in enumerate(impact):
                    rows.append({
                        "method": method,
                        "angle": angle,
                        "source_index": int(source_id[idx]),
                        "host": host,
                        "route_mass": float(mass[idx]),
                        "ablation_impact": float(value),
                    })
    raw = pd.DataFrame(rows)
    summary = (
        raw.groupby(["method", "angle", "host"])
        .agg(
            route_mass_mean=("route_mass", "mean"),
            ablation_mean=("ablation_impact", "mean"),
            ablation_median=("ablation_impact", "median"),
            positive_fraction=("ablation_impact", lambda values: float(np.mean(values > 0))),
            count=("ablation_impact", "count"),
        )
        .reset_index()
    )
    return raw, summary

host_raw_tables = []
host_summary_tables = []
for method, model in models.items():
    raw, summary = host_bundle(model, method, train_angles + interp_angles, test_indices)
    host_raw_tables.append(raw)
    host_summary_tables.append(summary)

host_raw_df = pd.concat(host_raw_tables, ignore_index=True)
host_summary_df = pd.concat(host_summary_tables, ignore_index=True)
host_summary_df.head()

## 12. Signed causal route-direction probe

The probe reports a signed effect along the learned route-angle direction, matched-norm orthogonal controls, a causal specificity ratio, and class-evidence change. Qualification still requires source-block confidence intervals, identity gates, and replication across seeds.

In [ ]:
def orthogonal_directions(tangent, count, seed):
    rng = np.random.default_rng(seed)
    directions = []
    while len(directions) < count:
        vector = rng.normal(size=tangent.shape)
        vector = vector - np.dot(vector, tangent) * tangent
        norm = np.linalg.norm(vector)
        if norm > 1e-12:
            directions.append(vector / norm)
    return directions


@torch.no_grad()
def causal_probe(model, method, trace, probe_angle=15.0, scales=(-2, -1, 0, 1, 2)):
    trained = trace[trace.angle.isin(train_angles)]
    contexts = stack_column(trained, "context").astype(np.float64)
    angles = trained.angle.to_numpy(float)
    x = contexts - contexts.mean(0, keepdims=True)
    y = angles - angles.mean()
    ridge = 1e-6 * max(np.trace(x.T @ x) / max(x.shape[1], 1), 1.0)
    tangent = np.linalg.solve(x.T @ x + ridge * np.eye(x.shape[1]), x.T @ y)
    tangent = tangent / max(np.linalg.norm(tangent), 1e-12)

    route_centroids = centroid_by_angle(trained, "route")
    route_matrix = np.stack([
        np.sqrt(np.clip(route_centroids[angle], 1e-12, None))
        for angle in train_angles
    ])
    centered_angles = np.asarray(train_angles) - np.mean(train_angles)
    route_centered = route_matrix - route_matrix.mean(0, keepdims=True)
    route_direction = (
        centered_angles[:, None] * route_centered
    ).sum(axis=0) / max(np.sum(centered_angles ** 2), 1e-12)
    route_direction = route_direction / max(np.linalg.norm(route_direction), 1e-12)

    orthogonals = orthogonal_directions(tangent, count=8, seed=SEED + 9)
    loader = fixed_loader(probe_angle, False, test_indices, False, batch_size=128)
    x_batch, y_batch, _, _ = next(iter(loader))
    x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
    base = model(x_batch)
    base_logp = F.log_softmax(base.logits, -1).gather(1, y_batch[:, None]).squeeze(1)

    tangent_tensor = torch.tensor(tangent, dtype=base.context.dtype, device=DEVICE)
    route_dir_tensor = torch.tensor(route_direction, dtype=base.route.dtype, device=DEVICE)

    rows = []
    for scale in scales:
        new_context = base.context + float(scale) * tangent_tensor
        new_route = torch.softmax(model.router(torch.cat([base.z0, new_context], dim=-1)), dim=-1)
        delta_root = torch.sqrt(new_route.clamp_min(1e-12)) - torch.sqrt(base.route.clamp_min(1e-12))
        signed_effect = float((delta_root * route_dir_tensor).sum(dim=1).mean().detach().cpu())

        z = torch.einsum("bh,bhd->bd", new_route, base.host_outputs)
        logits = model.classifier(model.synthesis_norm(z))
        new_logp = F.log_softmax(logits, -1).gather(1, y_batch[:, None]).squeeze(1)
        identity_drop = float((base_logp - new_logp).mean().detach().cpu())

        orth_effects = []
        for direction in orthogonals:
            direction_tensor = torch.tensor(direction, dtype=base.context.dtype, device=DEVICE)
            c_orth = base.context + float(scale) * direction_tensor
            r_orth = torch.softmax(model.router(torch.cat([base.z0, c_orth], dim=-1)), dim=-1)
            delta_orth = torch.sqrt(r_orth.clamp_min(1e-12)) - torch.sqrt(base.route.clamp_min(1e-12))
            orth_effects.append(float(torch.abs((delta_orth * route_dir_tensor).sum(dim=1)).mean().cpu()))

        orth_mean = float(np.mean(orth_effects)) if orth_effects else float("nan")
        csr = abs(signed_effect) / max(orth_mean, 1e-12) if scale != 0 else 0.0
        rows.append({
            "method": method,
            "probe_angle": probe_angle,
            "scale": scale,
            "signed_route_effect": signed_effect,
            "orthogonal_abs_effect_mean": orth_mean,
            "causal_specificity_ratio": csr,
            "class_log_prob_drop": identity_drop,
            "status": "development_not_qualified",
        })
    return pd.DataFrame(rows)

causal_df = pd.concat(
    [causal_probe(model, method, trace_df[trace_df.method == method]) for method, model in models.items()],
    ignore_index=True,
)
causal_df

## 13. Attribution, retention, and development gate summary

The summary separates two questions:

- **Geometry attribution:** paired geometry versus the compute-matched paired baseline.
- **Retention trade-off:** paired geometry with versus without the classification anchor.

A development row may be promising, neutral, or negative. It remains `not_qualified` until thresholds are frozen and multi-seed qualification is complete.

In [ ]:
def method_summary(method):
    route_row = geometry_df[(geometry_df.method == method) & (geometry_df.space == "route")].iloc[0]
    context_row = geometry_df[(geometry_df.method == method) & (geometry_df.space == "context")].iloc[0]
    synthesis_row = geometry_df[(geometry_df.method == method) & (geometry_df.space == "synthesis")].iloc[0]
    interp = interpolation_df[interpolation_df.method == method]
    forgetting = forgetting_df[forgetting_df.method == method]
    final_stage = int(evaluation_df.stage.max())
    final_trained = evaluation_df[
        (evaluation_df.method == method)
        & (evaluation_df.stage == final_stage)
        & (evaluation_df.angle_type == "train")
    ]
    csr = causal_df[(causal_df.method == method) & (causal_df.scale != 0)].causal_specificity_ratio.mean()
    compute = compute_summary_df[compute_summary_df.method == method].iloc[0]
    return {
        "method": method,
        "route_source_rho": route_row.source_rho_mean,
        "route_rho_ci_low": route_row.source_rho_ci_low,
        "route_centroid_rho": route_row.centroid_rho,
        "context_source_rho": context_row.source_rho_mean,
        "synthesis_source_rho": synthesis_row.source_rho_mean,
        "mean_interpolation_accuracy": interp.accuracy.mean(),
        "route_interp_win_fraction": interp.route_interpolation_beats_nearest.mean(),
        "mean_forgetting": forgetting.forgetting.mean(),
        "final_trained_accuracy": final_trained.accuracy.mean(),
        "mean_causal_specificity_ratio": csr,
        "total_forward_images": int(compute.total_forward_images),
        "total_optimizer_steps": int(compute.total_optimizer_steps),
        "qualification_status": "not_qualified",
    }

method_summary_df = pd.DataFrame([method_summary(method) for method in METHODS])


def metric(method, name):
    return float(method_summary_df.loc[method_summary_df.method == method, name].iloc[0])

attribution_df = pd.DataFrame([
    {
        "contrast": "geometry_effect_at_matched_compute",
        "treatment": "paired_geometry",
        "control": "paired_compute_matched_no_geometry",
        "delta_route_source_rho": metric("paired_geometry", "route_source_rho") - metric("paired_compute_matched_no_geometry", "route_source_rho"),
        "delta_context_source_rho": metric("paired_geometry", "context_source_rho") - metric("paired_compute_matched_no_geometry", "context_source_rho"),
        "delta_synthesis_source_rho": metric("paired_geometry", "synthesis_source_rho") - metric("paired_compute_matched_no_geometry", "synthesis_source_rho"),
        "delta_interpolation_accuracy": metric("paired_geometry", "mean_interpolation_accuracy") - metric("paired_compute_matched_no_geometry", "mean_interpolation_accuracy"),
        "delta_forgetting": metric("paired_geometry", "mean_forgetting") - metric("paired_compute_matched_no_geometry", "mean_forgetting"),
        "status": "development_not_qualified",
    },
    {
        "contrast": "retention_anchor_effect",
        "treatment": "paired_geometry_anchor_010",
        "control": "paired_geometry",
        "delta_route_source_rho": metric("paired_geometry_anchor_010", "route_source_rho") - metric("paired_geometry", "route_source_rho"),
        "delta_context_source_rho": metric("paired_geometry_anchor_010", "context_source_rho") - metric("paired_geometry", "context_source_rho"),
        "delta_synthesis_source_rho": metric("paired_geometry_anchor_010", "synthesis_source_rho") - metric("paired_geometry", "synthesis_source_rho"),
        "delta_interpolation_accuracy": metric("paired_geometry_anchor_010", "mean_interpolation_accuracy") - metric("paired_geometry", "mean_interpolation_accuracy"),
        "delta_forgetting": metric("paired_geometry_anchor_010", "mean_forgetting") - metric("paired_geometry", "mean_forgetting"),
        "status": "development_not_qualified",
    },
])

method_summary_df, attribution_df

## 14. Export and immutable manifest

All tables retain epistemic status. The manifest records the package version, git SHA, initial-state digest, split digests, compared methods, compute audit, sensory gate, and tracked changes.

In [ ]:
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

output_root = Path("results/v1_0_4") / f"seed_{SEED}"
output_root.mkdir(parents=True, exist_ok=True)

stage_df.to_csv(output_root / "stage_logs.csv", index=False)
compute_summary_df.to_csv(output_root / "compute_summary.csv", index=False)
evaluation_df.to_csv(output_root / "staged_evaluation.csv", index=False)
forgetting_df.to_csv(output_root / "forgetting.csv", index=False)
geometry_df.to_csv(output_root / "paired_geometry_metrics.csv", index=False)
interpolation_df.to_csv(output_root / "interpolation_metrics.csv", index=False)
host_summary_df.to_csv(output_root / "host_bundle_summary.csv", index=False)
host_raw_df.to_csv(output_root / "host_bundle_raw.csv", index=False)
causal_df.to_csv(output_root / "causal_probe.csv", index=False)
method_summary_df.to_csv(output_root / "development_method_summary.csv", index=False)
attribution_df.to_csv(output_root / "development_contrasts.csv", index=False)
pd.DataFrame(sensory_history).to_csv(output_root / "sensory_history.csv", index=False)
pd.DataFrame([sensory_gate_result]).to_csv(output_root / "sensory_gate.csv", index=False)

run_manifest = {
    "status": "development_not_qualified",
    "notebook_version": NOTEBOOK_VERSION,
    "package_version": geometry_mmalls.__version__,
    "git_sha": git_sha,
    "initial_mmalls_sha256": INITIAL_MMALS_SHA256,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": str(DEVICE),
    "method_specs": METHOD_SPECS,
    "train_angles": train_angles,
    "interpolation_angles": interp_angles,
    "extrapolation_angles": extra_angles,
    "split_manifest": split_manifest,
    "sensory_gate": {
        "threshold": SENSORY_GATE,
        "result": sensory_gate_result,
        "passed": sensory_gate_result["accuracy"] >= SENSORY_GATE,
    },
    "paired_compute_audit_passed": True,
    "tracked_changes": [f"CHG-104-{index:02d}" for index in range(1, 13)],
    "warning": (
        "This archive implements matched initialization, matched paired compute, "
        "and a retention-anchor control. It is not a qualified C1-C6 result."
    ),
    "timestamp": time.time(),
}
(output_root / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print("Exported to:", output_root.resolve())

## 15. Qualification checklist

Before any C1-C6 claim:

1. Freeze numeric gates before qualification runs.
2. Require the sensory acceptance gate to pass.
3. Run at least five seeds for pilot qualification and ten for final evidence.
4. Keep paired-control and paired-geometry initialization, data order, forwards, and optimizer steps identical.
5. Report failed seeds and all source-block confidence intervals.
6. Confirm held-out angles never enter training or checkpoint selection.
7. Add tuned EWC, replay, sparse-MoE, oracle-angle, and joint upper-bound controls for the broader CL claim.
8. Add cross-seed host-role matching before C3.
9. Require causal specificity confidence intervals and identity-preservation gates before C4.
10. Add memory-transport and dual-memory experiments before C5.
11. Replicate on a second controlled transformation or dataset.
12. Archive notebook, raw CSVs, manifest, environment, commit SHA, initial-state hash, and PDF report.

## 16. Optional clean PDF export

The separate archival protocol report under `docs/reports/` is the preferred GitHub document. Notebook export is optional. This notebook uses valid display-math delimiters and a list-style nbconvert command to avoid the earlier traitlets and LaTeX failures.

In [ ]:
# Optional: save or upload the notebook before running this cell.
# This cell is not required for the scientific run.
from pathlib import Path

patterns = [
    "Geometry_MMALS_G1_MatchedCompute_Retention_v1_0_4*.ipynb",
    "Geometry_MMALS_G1_CrossAngle_Paired_v1_0_3*.ipynb",
]
candidates = []
for pattern in patterns:
    candidates.extend(Path("/content").glob(pattern))

drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    for pattern in patterns:
        candidates.extend(drive_root.rglob(pattern))

candidates = sorted(
    {path.resolve() for path in candidates if path.is_file()},
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if candidates:
    notebook_path = candidates[0]
    output_dir = Path("/content/pdf_export")
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "pdf",
        str(notebook_path),
        "--output-dir",
        str(output_dir),
        "--PDFExporter.latex_command=xelatex",
        "--PDFExporter.latex_command={filename}",
        "--PDFExporter.latex_command=-interaction=nonstopmode",
        "--PDFExporter.latex_count=3",
    ]
    print(" ".join(command))
    # subprocess.run(command, check=True)  # Uncomment to export.
else:
    print("Notebook file not found under /content or Drive; PDF export skipped.")